# 1. Setup Mario

In [ ]:
# NOTE (2026): Updated for gymnasium + numpy 2.0 compatibility
# Old: !pip install gym_super_mario_bros==7.3.0 nes_py
%pip install gymnasium gym-super-mario-bros nes-py "stable-baselines3[extra]" torch matplotlib tqdm tensorboard rich


In [ ]:
# NOTE (2026): gym → gymnasium
import gymnasium as gym  # was `import gym`
import gym_super_mario_bros  # registers envs
# Import the Joypad wrapper
from nes_py.wrappers import JoypadSpace
# Import the SIMPLIFIED controls
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT


In [ ]:
# Setup game - NOTE (2026): gymnasium.make requires render_mode
env = gym.make('SuperMarioBros-v0', render_mode='rgb_array')
env = JoypadSpace(env, SIMPLE_MOVEMENT)


In [ ]:
# Create a flag - restart or not
done = True
# Loop through each frame in the game
for step in range(100000): 
    # Start the game to begin with 
    if done: 
        # Start the game - NOTE (2026): reset now returns (obs, info)
        obs, info = env.reset()
        state = obs
    # Do random actions - NOTE (2026): step now returns 5 values
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    state = obs
    done = terminated or truncated
    # Show the game on the screen - NOTE (2026): with rgb_array use matplotlib; human needs display
    # env.render()  # uncomment only with render_mode='human'
# Close the game
env.close()


# 2. Preprocess Environment

In [ ]:
# NOTE (2026): Updated torch install - CUDA wheel no longer pinned
# Old: !pip install torch==1.10.1+cu113 torchvision==0.11.2+cu113 torchaudio===0.10.1+cu113 -f https://download.pytorch.org/whl/cu113/torch_stable.html
%pip install torch


In [ ]:
# Install stable baselines for RL stuff - NOTE (2026): SB3 2.x is gymnasium-native
%pip install "stable-baselines3[extra]"


In [ ]:
# Import Frame Stacker Wrapper and GrayScaling Wrapper
# NOTE (2026): gym.wrappers.GrayScaleObservation → gymnasium.wrappers.GrayscaleObservation
from gymnasium.wrappers import GrayscaleObservation  # was gym.wrappers.GrayScaleObservation
# Import Vectorization Wrappers
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv
# Import Matplotlib to show the impact of frame stacking
from matplotlib import pyplot as plt


In [ ]:
# 1. Create the base environment
env = gym.make('SuperMarioBros-v0', render_mode='rgb_array')  # NOTE (2026): added render_mode
# 2. Simplify the controls 
env = JoypadSpace(env, SIMPLE_MOVEMENT)
# 3. Grayscale - NOTE (2026): spelling fixed
env = GrayscaleObservation(env, keep_dim=True)  # was GrayScaleObservation
# 4. Wrap inside the Dummy Environment
env = DummyVecEnv([lambda: env])
# 5. Stack the frames
env = VecFrameStack(env, 4, channels_order='last')


In [ ]:
state = env.reset()

In [ ]:
# NOTE (2026): VecEnv step still returns 4 values (wraps gymnasium 5-tuple)
state, reward, done, info = env.step([5])


In [ ]:
plt.figure(figsize=(20,16))
for idx in range(state.shape[3]):
    plt.subplot(1,4,idx+1)
    plt.imshow(state[0][:,:,idx])
plt.show()

# 3. Train the RL Model

In [ ]:
# Import os for file path management
import os 
# Import PPO for algos
from stable_baselines3 import PPO
# Import Base Callback for saving models
from stable_baselines3.common.callbacks import BaseCallback

In [ ]:
class TrainAndLoggingCallback(BaseCallback):

    def __init__(self, check_freq, save_path, verbose=1):
        super(TrainAndLoggingCallback, self).__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path

    def _init_callback(self):
        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            model_path = os.path.join(self.save_path, 'best_model_{}'.format(self.n_calls))
            self.model.save(model_path)

        return True

In [ ]:
CHECKPOINT_DIR = './train/'
LOG_DIR = './logs/'

In [ ]:
# Setup model saving callback
callback = TrainAndLoggingCallback(check_freq=10000, save_path=CHECKPOINT_DIR)

In [ ]:
# This is the AI model started
# NOTE (2026): handle missing tensorboard gracefully
try:
    import tensorboard
    tb_log = LOG_DIR
except ImportError:
    tb_log = None
model = PPO('CnnPolicy', env, verbose=1, tensorboard_log=tb_log, learning_rate=0.000001, 
            n_steps=512) 


In [ ]:
# Train the AI model, this is where the AI model starts to learn
try:
    model.learn(total_timesteps=1000000, callback=callback, progress_bar=True)
except ImportError:
    model.learn(total_timesteps=1000000, callback=callback)


In [ ]:
model.save('thisisatestmodel')

# 4. Test it Out

In [ ]:
# Load model - NOTE (2026): ensure checkpoint exists (1M steps creates best_model_1000000)
model = PPO.load('./train/best_model_1000000')


In [ ]:
state = env.reset()

In [ ]:
# Start the game 
state = env.reset()
# Loop through the game
while True: 
    
    action, _ = model.predict(state)
    state, reward, done, info = env.step(action)  # VecEnv 4-tuple (raw gymnasium would be 5)
    # env.render()  # NOTE (2026): use render_mode='human' locally, or matplotlib with rgb_array
